# ML-PD：最小可复现实验

原始合作数据未公开。本 Notebook 使用合成数值数据，验证程序流程，不产生临床结论。先按仓库 README 安装 Python 3.12 环境，并为该环境选择 Jupyter kernel。

新版核心逻辑在 `src/mlpd`，此文件只负责调用，避免多个 Notebook 复制不同版本的实验代码。

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
assert (root / "pyproject.toml").exists(), "Open this notebook from the repository or notebooks directory."
run_name = datetime.now().strftime("notebook-%Y%m%d-%H%M%S")
data_dir = root / "data" / run_name
out_dir = root / "runs" / run_name

subprocess.run([sys.executable, "-m", "mlpd", "demo", "--output", str(data_dir)], check=True, cwd=root)

## 实际执行

所有预处理与特征选择在训练折内学习。此处以先验基线与逻辑回归运行四个任务，使用 3 外折、2 内折。完整模型对照见[运行指南](../guides/quickstart.md)。

In [ ]:
subprocess.run([sys.executable, "-m", "mlpd", "run",
    "--data", str(data_dir / "synthetic.csv"), "--schema", str(data_dir / "schema.json"),
    "--output", str(out_dir), "--models", "dummy", "logistic",
    "--outer-folds", "3", "--inner-folds", "2"], check=True, cwd=root)

In [ ]:
import pandas as pd
report = json.loads((out_dir / "report.json").read_text(encoding="utf-8"))
pd.DataFrame([{ "task": task["id"], "model": model["name"],
    "balanced_accuracy_mean": model["summary"]["balanced_accuracy"]["mean"],
    "fold_sd": model["summary"]["balanced_accuracy"]["std"] }
    for task in report["tasks"] for model in task["models"]])

In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(out_dir / "figures" / "three-class-comparison.svg")))

## 如何解释

折间 SD 不是置信区间；合成特征没有代谢物身份；横断面分类不等于疾病发展预测。真实数据接入前请阅读 [数据契约](../guides/data-contract.md)，结果解释请阅读 [方法说明](../guides/methodology.md)。